# 12 · Capstone: Code/AST as Graphs for Vulnerability Detection

## What this notebook covers
The capstone builds on every technique in this series to implement a GNN-based vulnerability detection pipeline — the `semantic-vuln-chain-analysis` prototype from the backlog.

## System design
```
Source code → AST parsing → Graph construction
                                    ↓
                     Node features (token type, depth, ...)
                                    ↓
                       GNN (GCN / GIN) classification
                                    ↓
                    Vulnerability prediction per function
```

## Why ASTs as graphs?
An Abstract Syntax Tree (AST) is the natural graph representation of code. Each node is a code element (function call, identifier, literal, operator); each edge is a structural relationship (child, sibling, control flow). GNNs on ASTs can capture:
- Structural patterns like `strcpy(buf, user_input)` without fixed string matching
- Data flow from user input to dangerous sinks across multiple function calls
- Context-sensitive patterns that regex/static rules miss

This is the foundation for tools like **CodeBERT** (which uses transformer + AST), **Devign** (GNN on control-flow graphs), and `semantic-vuln-chain-analysis` in the backlog.

In [ ]:
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import json
torch.manual_seed(42); np.random.seed(42)

try:
    from torch_geometric.data import Data, Batch
    from torch_geometric.nn import GINConv, global_add_pool
    HAS_PYG = True
except ImportError:
    HAS_PYG = False

print("AST → Graph → GNN vulnerability detector")

In [ ]:
# ── AST to graph ───────────────────────────────────────────────────────────────
def ast_to_graph(source_code):
    """
    Parse Python source code into a graph.
    Nodes = AST nodes; edges = parent→child relationships.
    Node features: one-hot node type.
    """
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return None, None, None

    node_types = []
    edges = []
    node_counter = [0]

    all_types = sorted(set(type(n).__name__ for n in ast.walk(tree)))
    type_to_idx = {t: i for i, t in enumerate(all_types)}

    def walk(node, parent_id=None):
        my_id = node_counter[0]
        node_counter[0] += 1
        node_types.append(type_to_idx.get(type(node).__name__, 0))
        if parent_id is not None:
            edges.append((parent_id, my_id))
        for child in ast.iter_child_nodes(node):
            walk(child, my_id)

    walk(tree)
    return node_types, edges, len(all_types)

# Test on a sample
sample_code = """
def process_input(user_data):
    buf = user_data[:100]
    result = eval(buf)
    return result
"""
node_types, edges, n_types = ast_to_graph(sample_code)
print(f"AST graph: {len(node_types)} nodes, {len(edges)} edges, {n_types} node types")

In [ ]:
# ── Synthetic dataset: vulnerable vs safe functions ──────────────────────────
VULNERABLE_FUNCTIONS = [
    "def vuln1(x):\n    cmd = 'echo ' + x\n    os.system(cmd)\n    return cmd",
    "def vuln2(data):\n    return eval(data)",
    "def vuln3(q):\n    db.execute('SELECT * FROM users WHERE id=' + q)",
    "def vuln4(path):\n    f = open(path)\n    return f.read()",
    "def vuln5(s):\n    import subprocess\n    subprocess.call(s, shell=True)",
    "def vuln6(cmd):\n    result = os.popen(cmd).read()\n    return result",
    "def vuln7(x):\n    exec(x)\n    return True",
    "def vuln8(t):\n    import pickle\n    return pickle.loads(t)",
]

SAFE_FUNCTIONS = [
    "def safe1(x):\n    return x.strip().lower()",
    "def safe2(data):\n    return json.loads(data)",
    "def safe3(q):\n    return db.execute('SELECT * FROM users WHERE id=?', (q,))",
    "def safe4(path):\n    if path.startswith('/allowed/'):\n        return open(path).read()",
    "def safe5(items):\n    return [x*2 for x in items if isinstance(x, int)]",
    "def safe6(n):\n    return [i**2 for i in range(n)]",
    "def safe7(s):\n    return hashlib.sha256(s.encode()).hexdigest()",
    "def safe8(data):\n    return base64.b64decode(data).decode('utf-8')",
]

def build_pyg_data(code, label):
    """Convert code string to PyG Data object."""
    node_types, edges, n_types = ast_to_graph(code)
    if node_types is None or len(node_types) < 2:
        return None
    x = F.one_hot(torch.LongTensor(node_types), num_classes=max(n_types, 50)).float()
    if edges:
        edge_index = torch.LongTensor(edges).T
        # Add reverse edges
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=torch.LongTensor([label]))

dataset = []
for code in VULNERABLE_FUNCTIONS:
    d = build_pyg_data(code, 1)
    if d: dataset.append(d)
for code in SAFE_FUNCTIONS:
    d = build_pyg_data(code, 0)
    if d: dataset.append(d)

print(f"Dataset: {len(dataset)} graphs  |  {sum(d.y.item() for d in dataset)} vulnerable")

In [ ]:
if HAS_PYG:
    from torch_geometric.loader import DataLoader as PyGLoader

    # Train/test split
    np.random.shuffle(dataset)
    n_tr = int(len(dataset) * 0.75)
    train_ds, test_ds = dataset[:n_tr], dataset[n_tr:]

    in_dim = dataset[0].x.shape[1]

    class VulnGNN(nn.Module):
        def __init__(self, in_dim, hidden=32):
            super().__init__()
            mlp1 = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            mlp2 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.conv1 = GINConv(mlp1)
            self.conv2 = GINConv(mlp2)
            self.head  = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 2))

        def forward(self, x, edge_index, batch):
            x = F.relu(self.conv1(x, edge_index))
            x = F.relu(self.conv2(x, edge_index))
            x = global_add_pool(x, batch)
            return self.head(x)

    model = VulnGNN(in_dim)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loader = PyGLoader(dataset, batch_size=4, shuffle=True)

    for epoch in range(100):
        model.train()
        for batch in loader:
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for d in dataset:
            out = model(d.x, d.edge_index, torch.zeros(d.x.size(0), dtype=torch.long))
            if out.argmax(dim=1).item() == d.y.item():
                correct += 1
    print(f"GNN vulnerability detector accuracy: {correct}/{len(dataset)} = {correct/len(dataset):.2f}")
else:
    # Manual GIN-like aggregation for demonstration
    print("PyTorch Geometric not available — running rule-based baseline for comparison.")

    DANGEROUS_CALLS = {"eval", "exec", "os.system", "subprocess.call", "popen", "pickle.loads"}

    def rule_based_detector(code):
        return int(any(kw in code for kw in DANGEROUS_CALLS))

    preds = [rule_based_detector(c) for c in VULNERABLE_FUNCTIONS + SAFE_FUNCTIONS]
    true  = [1]*len(VULNERABLE_FUNCTIONS) + [0]*len(SAFE_FUNCTIONS)
    acc   = sum(p==t for p,t in zip(preds, true)) / len(true)
    print(f"Rule-based baseline accuracy: {acc:.2f}")
    print("GNN improves on this by learning structural patterns beyond keyword matching.")

In [ ]:
# ── Visualise two ASTs: vulnerable vs safe ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (code, title) in zip(axes, [
    (VULNERABLE_FUNCTIONS[1], "VULNERABLE: eval(data)"),
    (SAFE_FUNCTIONS[1],       "SAFE: json.loads(data)")
]):
    node_types, edges, n_types = ast_to_graph(code)
    if not edges: continue
    G_ast = nx.DiGraph()
    G_ast.add_nodes_from(range(len(node_types)))
    G_ast.add_edges_from(edges)
    pos = nx.spring_layout(G_ast, seed=42)
    colour = "tomato" if "VULNERABLE" in title else "steelblue"
    nx.draw(G_ast, pos=pos, ax=ax, node_size=60, node_color=colour,
            edge_color="grey", arrows=True, alpha=0.7, with_labels=False)
    ax.set_title(f"{title}\n({len(node_types)} nodes, {len(edges)} edges)")

plt.suptitle("AST Graphs: Vulnerable vs Safe Code", fontsize=13)
plt.tight_layout()
plt.savefig(f"{base}/12_ast_gnn_capstone.png", dpi=120)
plt.show()

## Production extension notes
This prototype demonstrates the core pipeline. A production `semantic-vuln-chain-analysis` system would extend this with:

1. **Control flow graphs (CFG)** in addition to ASTs — capturing execution paths, not just syntax
2. **Data flow graphs (DFG)** — tracking taint propagation from user input to dangerous sinks
3. **Inter-procedural analysis** — connecting call sites to function definitions across files
4. **Pretrained code embeddings** (CodeBERT, UniXcoder) as node features instead of one-hot AST types
5. **Dataflow-aware edge types** — different edge types for syntactic, control, and data dependencies

The GNN architecture scales naturally to all three: heterogeneous GNNs (NB 10) handle multiple edge types; GIN's sum aggregation is optimal for counting structural patterns.